# Approach 1 — Fit on averaged raw CE

**Pipeline:** Average raw CE_TEST across 100 runs at each batch number → fit one curve `CE(x) = A + B/(x+1)^n` → compute IPA from that single fit.

**Input:** `prune_layers_ALL/p-percentage_{p}/batch_size_{bs}/averaged_runs_p_{p}_bs_{bs}.csv` (column `Avg_CE_Test`).

**Output:** `intermediate/approach_1_fit_params_bs_{bs}.csv` with every `P%, A, B, n, CE_o, CE_L, learn_BN, IPA` row, plus `ipa_summary_approach_1.csv`.

**IPA:** `abs(CE_o - CE_L) / learn_BN` where `CE_L = CE_o - 0.9*(CE_o - A)`.

In [5]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
FIT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\Fitting_IPA_curves_data_I"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")

# Fit-function helpers (verbatim from fitting_function_IPA.ipynb)
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0, 1000
N_MIN, N_MAX = 0.5, 3.0


def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0


def model(params, x):
    vals = params.valuesdict()
    A, B, n = vals["A"], vals["B"], vals["n"]
    return A + B / ((x + 1) ** n)


def residual(params, x, data):
    weight = x
    return weight * (model(params, x) - data)


def fit_curve(x, y):
    mask  = ~np.isnan(y)
    x_fit = np.asarray(x)[mask]
    y_fit = np.asarray(y)[mask]
    if len(x_fit) < 10:
        return None
    A0, n0, B0 = initialize_guesses(x_fit, y_fit)
    params = Parameters()
    params.add("A", value=A0, min=A_MIN, max=A_MAX)
    params.add("B", value=B0, min=B_MIN, max=B_MAX)
    params.add("n", value=n0, min=N_MIN, max=N_MAX)
    try:
        return minimize(residual, params, args=(x_fit, y_fit))
    except Exception:
        return None

# --- IPA: learn_BN from avg data; analytical fallback when data ends early ---
# CE_L = CE_o - 0.9*(CE_o - A)   — derived from fitted A
# IPA  = abs(CE_o - CE_L) / learn_BN
#
# learn_BN priority:
#   1. Averaged data: first BN where Avg_CE_Test <= CE_L
#      avg_CE_learn_at_BN = that raw CE value
#   2. Analytical fallback (data ends before crossing CE_L):
#      learn_BN = ceil((B / (CE_L - A))^(1/n) - 1)
#      avg_CE_learn_at_BN = NaN  (BN is beyond the recorded data)
def compute_ipa_from_fit(x_grid, ce_data, A, B, n):
    x_grid  = np.asarray(x_grid,  dtype=float)
    ce_data = np.asarray(ce_data, dtype=float)
    CE_L = CE_o - 0.9 * (CE_o - A)
    mask = ce_data <= CE_L

    if mask.any():
        learn_BN        = float(x_grid[mask][0])
        avg_CE_at_learn = float(ce_data[mask][0])
    else:
        denom = CE_L - A
        if denom <= 0 or n <= 0 or B <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "avg_CE_learn_at_BN": np.nan}
        BN_analytic = (B / denom) ** (1.0 / n) - 1.0
        if not np.isfinite(BN_analytic) or BN_analytic <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "avg_CE_learn_at_BN": np.nan}
        learn_BN        = float(np.ceil(BN_analytic))
        avg_CE_at_learn = np.nan   # BN is beyond the recorded data

    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "avg_CE_learn_at_BN": avg_CE_at_learn}
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "avg_CE_learn_at_BN": avg_CE_at_learn}
print("Cell 1 ready.")

Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
Cell 1 ready.


In [6]:
# === Cell 2 — Approach 1: fit averaged raw CE for A,B,n; find learn_BN from raw averaged data ===
# Per (P%, BS): load averaged_runs_p_{p}_bs_{bs}.csv, fit Avg_CE_Test to get A,B,n,
# then find learn_BN as the first BN where the raw Avg_CE_Test <= CE_L (not the fitted curve).
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 1 — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {avg_csv}")
            continue
        df = pd.read_csv(avg_csv)
        df.columns = df.columns.str.strip()
        ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  — unexpected columns {list(df.columns)}")
            continue
        df = df.dropna(subset=[ce_col, bn_col])
        x = df[bn_col].values.astype(float)
        y = df[ce_col].values.astype(float)

        result = fit_curve(x, y)
        if result is None:
            print(f"  [FAIL] P%={p*100:5.1f}%  — fit did not converge")
            continue
        A = result.params["A"].value
        B = result.params["B"].value
        n = result.params["n"].value
        ipa = compute_ipa_from_fit(x, y, A, B, n)

        src = "data" if np.isfinite(ipa["avg_CE_learn_at_BN"]) else "analytic"
        print(f"  P%={p*100:5.1f}%  A={A:.4f}  B={B:.4f}  n={n:.4f}  "
              f"CE_o={CE_o:.4f}  CE_L={ipa['CE_L']:.4f}  "
              f"learn_BN={ipa['learn_BN']!r:>8}  IPA={ipa['IPA']}  [{src}]")

        rows.append({
            "P%": p * 100, "A": A, "B": B, "n": n,
            "CE_o": CE_o, "CE_L": ipa["CE_L"],
            "learn_BN": ipa["learn_BN"], "avg_CE_learn_at_BN": ipa["avg_CE_learn_at_BN"],
            "IPA": ipa["IPA"],
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_1_fit_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")


  Approach 1 — Batch size 64


IndexError: boolean index did not match indexed array along axis 0; size of axis is 290 but size of corresponding boolean axis is 340

In [ ]:
# === Cell 3 — Build wide summary CSV for Approach 1 ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches; populated only by 2b.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val, std_val = np.nan, np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if (not sub.empty and "IPA" in sub.columns) else (
                       float(sub["IPA_mean"].iloc[0]) if (not sub.empty and "IPA_mean" in sub.columns) else np.nan)
        std_val  = np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = std_val
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_1.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))


In [4]:
# === Cell 4 — Plot IPA vs P% ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG   = "1"
TITLE = "Approach 1 — Fit on averaged raw CE"
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(10, 6))

for bs in BATCH_SIZES:
    mean_col = f"IPA_Avg_{bs}"
    sub = summary_df.dropna(subset=[mean_col])
    if sub.empty:
        continue
    ax.plot(sub["P%"].values, sub[mean_col].values,
            label=f"BS={bs}", color=BS_COLOR[bs], marker="o", markersize=6, linewidth=2)

ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
# ax.set_yscale("log")
ax.set_title(TITLE, fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

out_png = os.path.join(OUT_DIR, f"ipa_plot_approach_{TAG}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\ipa_plot_approach_1.png
